# 🐶🐱 Hond / Kat / Overige Classifier

Dit notebook classificeert afbeeldingen in drie categorieën:
- 🐶 **Hond**
- 🐱 **Kat**
- ❓ **Overige** (geen hond of kat)

We gebruiken **MobileNetV2** — een voorgetraind neuraal netwerk van ImageNet (1.2 miljoen afbeeldingen, 1000 klassen). We hoeven het netwerk niet zelf te trainen!

---
### Hoe werkt het?
1. We laden een afbeelding
2. Het model geeft een lijst van 1000 mogelijke klassen terug met een kans per klasse
3. We kijken of een van de top-klassen een **hond** of **kat** breed is
4. Als dat niet zo is → **Overige**

## Stap 1 — Bibliotheken installeren & importeren

In [ ]:
# Installeer benodigde pakketten (eenmalig)
# Verwijder het # hieronder als je dit nog niet geïnstalleerd hebt
# !pip install torch torchvision pillow matplotlib requests

import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
import json
from io import BytesIO
import os

print('✅ Alle imports geslaagd!')
print(f'   PyTorch versie: {torch.__version__}')

## Stap 2 — Model laden

We laden **MobileNetV2**, een lichtgewicht maar krachtig neuraal netwerk. De eerste keer download het de gewichten (~14 MB).

In [ ]:
print('⏳ Model laden... (eerste keer downloadt het ~14 MB)')

# Laad voorgetraind MobileNetV2 model
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
model.eval()  # Zet in evaluatie-modus (niet trainen)

# Laad de ImageNet klasse-labels (1000 klassen)
LABELS_URL = 'https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json'
try:
    labels = json.loads(requests.get(LABELS_URL).text)
    print(f'✅ Model geladen! ({len(labels)} klassen bekend)')
except:
    # Fallback: gebruik de ingebouwde labels van torchvision
    labels = models.MobileNet_V2_Weights.IMAGENET1K_V1.meta['categories']
    print(f'✅ Model geladen met ingebouwde labels ({len(labels)} klassen)')

## Stap 3 — Classificatie-logica

Hier definiëren we welke ImageNet-klassen we als 'hond' of 'kat' beschouwen, en de functies om een afbeelding te analyseren.

In [ ]:
# ImageNet klasse-indices voor honden (151 t/m 268) en katten (281 t/m 285)
# Dit zijn de officiële ranges in het ImageNet-schema
HOND_RANGE = range(151, 269)   # 118 hondenrassen
KAT_RANGE  = range(281, 286)   # 5 kattenklassen

# Afbeelding voorbereiden voor het model
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

def classificeer_afbeelding(img: Image.Image, drempel_overige: float = 0.30):
    """
    Classificeert een PIL-afbeelding als Hond, Kat of Overige.
    
    Parameters:
        img               : PIL Image object
        drempel_overige   : minimale gecombineerde kans voor hond/kat om niet als 'Overige' te gelden
    
    Returns:
        dict met: label, kans, top5, hond_kans, kat_kans
    """
    # Zorg dat afbeelding RGB is
    img_rgb = img.convert('RGB')
    
    # Verwerk afbeelding
    input_tensor = preprocess(img_rgb).unsqueeze(0)  # batch van 1
    
    # Voorspelling
    with torch.no_grad():
        output = model(input_tensor)
    
    # Kansen berekenen (softmax)
    probs = torch.nn.functional.softmax(output[0], dim=0)
    
    # Top-5 voorspellingen
    top5_probs, top5_indices = torch.topk(probs, 5)
    top5 = [
        {'label': labels[idx], 'kans': float(prob)}
        for prob, idx in zip(top5_probs, top5_indices)
    ]
    
    # Hond- en kat-kansen optellen
    hond_kans = float(sum(probs[i] for i in HOND_RANGE))
    kat_kans  = float(sum(probs[i] for i in KAT_RANGE))
    
    # Beslissing
    if hond_kans < drempel_overige and kat_kans < drempel_overige:
        label = 'Overige ❓'
        kans  = 1.0 - hond_kans - kat_kans
    elif hond_kans >= kat_kans:
        label = 'Hond 🐶'
        kans  = hond_kans
    else:
        label = 'Kat 🐱'
        kans  = kat_kans
    
    return {
        'label':     label,
        'kans':      kans,
        'top5':      top5,
        'hond_kans': hond_kans,
        'kat_kans':  kat_kans,
    }

def laad_afbeelding_van_url(url: str) -> Image.Image:
    """Laad een afbeelding van een URL."""
    response = requests.get(url, timeout=10)
    return Image.open(BytesIO(response.content))

def laad_afbeelding_van_pad(pad: str) -> Image.Image:
    """Laad een afbeelding van een lokaal bestandspad."""
    return Image.open(pad)

print('✅ Classificatie-functies klaar!')
print(f'   Hond-klassen: {len(HOND_RANGE)} rassen (indices {HOND_RANGE.start}–{HOND_RANGE.stop-1})')
print(f'   Kat-klassen:  {len(KAT_RANGE)} klassen (indices {KAT_RANGE.start}–{KAT_RANGE.stop-1})')

## Stap 4 — Eén afbeelding classificeren

Test met een afbeelding van een URL, of verander de URL naar een eigen afbeelding.

In [ ]:
def toon_resultaat(img, resultaat):
    """Toon afbeelding met classificatieresultaat."""
    kleur_map = {'Hond 🐶': '#e07b54', 'Kat 🐱': '#7b9ed9', 'Overige ❓': '#888888'}
    kleur = kleur_map.get(resultaat['label'], '#cccccc')
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Linksː afbeelding
    ax1.imshow(img)
    ax1.axis('off')
    ax1.set_title(
        f"{resultaat['label']}  —  {resultaat['kans']*100:.1f}% zekerheid",
        fontsize=16, fontweight='bold', color=kleur, pad=12
    )
    
    # Rechts: staafdiagram van top-5
    namen  = [r['label'] for r in resultaat['top5']]
    kansen = [r['kans'] * 100 for r in resultaat['top5']]
    bars   = ax2.barh(namen[::-1], kansen[::-1], color='#cccccc')
    bars[-1].set_color(kleur)  # Highlight top-1
    ax2.set_xlabel('Kans (%)')
    ax2.set_title('Top-5 voorspellingen', fontsize=13)
    ax2.set_xlim(0, max(kansen) * 1.25)
    for bar, k in zip(bars, kansen[::-1]):
        ax2.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{k:.1f}%', va='center', fontsize=10)
    
    # Hond/Kat totaalkansen
    fig.text(0.5, 0.01,
             f"Totale hond-kans: {resultaat['hond_kans']*100:.1f}%   |   "
             f"Totale kat-kans: {resultaat['kat_kans']*100:.1f}%",
             ha='center', fontsize=10, color='gray')
    
    plt.tight_layout()
    plt.show()

# -------------------------------------------------------
# 🔧 AANPASSEN: verander de URL of gebruik een lokaal pad
# -------------------------------------------------------
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/1200px-YellowLabradorLooking_new.jpg'
# url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/b/bb/Kittyply_edit1.jpg/1200px-Kittyply_edit1.jpg'
# pad = 'mijn_foto.jpg'  # of: laad_afbeelding_van_pad(pad)

img = laad_afbeelding_van_url(url)
resultaat = classificeer_afbeelding(img)
toon_resultaat(img, resultaat)

## Stap 5 — Meerdere afbeeldingen tegelijk classificeren

Geef een lijst van URLs (of paden) mee en zie het resultaat als overzicht.

In [ ]:
# Lijst van afbeeldingen om te testen
test_afbeeldingen = [
    ('Labrador',  'https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/400px-YellowLabradorLooking_new.jpg'),
    ('Kat',       'https://upload.wikimedia.org/wikipedia/commons/thumb/b/bb/Kittyply_edit1.jpg/400px-Kittyply_edit1.jpg'),
    ('Panda',     'https://upload.wikimedia.org/wikipedia/commons/thumb/0/0f/Grosser_Panda.JPG/400px-Grosser_Panda.JPG'),
    ('Auto',      'https://upload.wikimedia.org/wikipedia/commons/thumb/1/1b/2021_Toyota_GR_Supra_A90_facelift%2C_front_8.17.21.jpg/400px-2021_Toyota_GR_Supra_A90_facelift%2C_front_8.17.21.jpg'),
    ('Husky',     'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/400px-Camponotus_flavomarginatus_ant.jpg'),
]

resultaten = []
for naam, url in test_afbeeldingen:
    try:
        img = laad_afbeelding_van_url(url)
        r = classificeer_afbeelding(img)
        resultaten.append((naam, img, r))
        print(f'✅ {naam:15s} → {r["label"]:12s} ({r["kans"]*100:.1f}%)')
    except Exception as e:
        print(f'❌ {naam}: fout — {e}')

print(f'\nKlaar! {len(resultaten)} afbeeldingen verwerkt.')

In [ ]:
# Toon alle resultaten als grid
kleur_map = {'Hond 🐶': '#e07b54', 'Kat 🐱': '#7b9ed9', 'Overige ❓': '#888888'}
n = len(resultaten)
cols = min(3, n)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
axes = [axes] if n == 1 else axes.flatten() if rows > 1 else list(axes)

for ax, (naam, img, r) in zip(axes, resultaten):
    ax.imshow(img)
    ax.axis('off')
    kleur = kleur_map.get(r['label'], '#cccccc')
    ax.set_title(
        f"{naam}\n{r['label']} — {r['kans']*100:.0f}%",
        color=kleur, fontweight='bold', fontsize=11
    )
    # Gekleurde rand om afbeelding
    for spine in ax.spines.values():
        spine.set_edgecolor(kleur)
        spine.set_linewidth(3)
        spine.set_visible(True)

# Verberg lege vakjes
for ax in axes[n:]:
    ax.set_visible(False)

plt.suptitle('🐶 Hond / 🐱 Kat / ❓ Overige — Classificatie-overzicht', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Stap 6 — Eigen afbeeldingen uploaden

Gebruik dit blok om een **lokale afbeelding** van je computer te classificeren.

In [ ]:
# -------------------------------------------------------
# Methode A: lokaal bestandspad
# Verander 'mijn_foto.jpg' naar jouw bestandsnaam
# -------------------------------------------------------
pad = 'mijn_foto.jpg'

if os.path.exists(pad):
    img = laad_afbeelding_van_pad(pad)
    resultaat = classificeer_afbeelding(img)
    print(f'Resultaat: {resultaat["label"]} ({resultaat["kans"]*100:.1f}%)')
    toon_resultaat(img, resultaat)
else:
    print(f'⚠️  Bestand "{pad}" niet gevonden.')
    print('   Zet je afbeelding in dezelfde map als dit notebook,')
    print('   of verander het pad hierboven.')

## Stap 7 — Map met afbeeldingen verwerken

Classificeer **alle afbeeldingen** in een map en sla de resultaten op als CSV.

In [ ]:
import csv

# -------------------------------------------------------
# 🔧 AANPASSEN: verander naar jouw mapnaam
# -------------------------------------------------------
map_pad    = 'afbeeldingen'   # map met je foto's
uitvoer_csv = 'resultaten.csv'
extensies  = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

if not os.path.isdir(map_pad):
    print(f'⚠️  Map "{map_pad}" bestaat niet.')
    print('   Maak de map aan en zet er afbeeldingen in.')
    print('   Of pas "map_pad" aan naar de juiste locatie.')
else:
    bestanden = [
        f for f in os.listdir(map_pad)
        if os.path.splitext(f)[1].lower() in extensies
    ]
    print(f'📁 {len(bestanden)} afbeeldingen gevonden in "{map_pad}"')
    
    rijen = []
    for i, bestand in enumerate(bestanden, 1):
        try:
            img = laad_afbeelding_van_pad(os.path.join(map_pad, bestand))
            r = classificeer_afbeelding(img)
            rijen.append({
                'bestand':    bestand,
                'label':      r['label'],
                'kans':       f"{r['kans']*100:.1f}%",
                'hond_kans':  f"{r['hond_kans']*100:.1f}%",
                'kat_kans':   f"{r['kat_kans']*100:.1f}%",
                'top1_model': r['top5'][0]['label'],
            })
            print(f'  [{i}/{len(bestanden)}] {bestand:30s} → {r["label"]}')
        except Exception as e:
            print(f'  ❌ {bestand}: {e}')
    
    # Sla op als CSV
    with open(uitvoer_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=rijen[0].keys())
        writer.writeheader()
        writer.writerows(rijen)
    
    print(f'\n✅ Resultaten opgeslagen in "{uitvoer_csv}"')
    
    # Samenvatting
    from collections import Counter
    telling = Counter(r['label'] for r in rijen)
    print('\n📊 Samenvatting:')
    for label, aantal in telling.most_common():
        print(f'   {label}: {aantal}x')

---
## 💡 Tips & volgende stappen

**De `drempel_overige` parameter aanpassen:**
```python
resultaat = classificeer_afbeelding(img, drempel_overige=0.20)  # minder streng
resultaat = classificeer_afbeelding(img, drempel_overige=0.50)  # strenger
```

**Eigen afbeelding via URL:**
```python
url = 'https://jouw-url-hier.jpg'
img = laad_afbeelding_van_url(url)
resultaat = classificeer_afbeelding(img)
toon_resultaat(img, resultaat)
```

**Volgende stappen als je verder wil:**
- 🔁 **Fine-tuning**: train het model bij op jouw eigen dataset (transfer learning)
- 📊 **Evaluatie**: bereken accuracy, precision en recall op een testset  
- 🚀 **Groter model**: probeer ResNet50 of EfficientNet voor hogere nauwkeurigheid
- 🌐 **Webcam**: gebruik OpenCV om live video te classificeren